<a href="https://colab.research.google.com/github/gokulbytes/personalized-news-recommendation-engine/blob/main/main_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements

In [1]:
# Install necessary libraries
!pip install pandas numpy scikit-learn -q

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# main

## behaviors_df

In [3]:
# Load the dataset
behaviors_df = pd.read_csv('/content/behaviors.tsv', sep='\t', header=None, names=['UserID', 'TimeStamp', 'History', 'Impressions'])
behaviors_df.head()

,UserID,TimeStamp,History,Impressions
1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


In [4]:
behaviors_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 156965 entries, 1 to 156965
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   UserID       156965 non-null  object
 1   TimeStamp    156965 non-null  object
 2   History      153727 non-null  object
 3   Impressions  156965 non-null  object
dtypes: object(4)
memory usage: 6.0+ MB


### Drop Duplicates

In [5]:
# Remove duplicate rows based on all columns and reset the index
behaviors_df = behaviors_df.drop_duplicates().reset_index(drop=True)

## news_df

In [6]:
# Load the dataset
news_df = pd.read_csv('/content/news.tsv', sep='\t', header=None, names=['NewsID', 'Category', 'SubCategory', 'Title', 'Abstract', 'URL','TitleEntities','AbstractEntities'])
news_df.head()

,NewsID,Category,SubCategory,Title,Abstract,URL,TitleEntities,AbstractEntities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


In [7]:
news_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51282 entries, 0 to 51281
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   NewsID            51282 non-null  object
 1   Category          51282 non-null  object
 2   SubCategory       51282 non-null  object
 3   Title             51282 non-null  object
 4   Abstract          48616 non-null  object
 5   URL               51282 non-null  object
 6   TitleEntities     51279 non-null  object
 7   AbstractEntities  51278 non-null  object
dtypes: object(8)
memory usage: 3.1+ MB


### Drop Duplicates

In [8]:
# Remove duplicate rows based on all columns and reset the index
news_df = news_df.drop_duplicates().reset_index(drop=True)

## Data Preprocessing

### behaviors_df

In [9]:
# Fill missing values in the 'History' column with empty strings
behaviors_df['History'].fillna('', inplace=True)

behaviors_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156965 entries, 0 to 156964
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   UserID       156965 non-null  object
 1   TimeStamp    156965 non-null  object
 2   History      156965 non-null  object
 3   Impressions  156965 non-null  object
dtypes: object(4)
memory usage: 4.8+ MB


### news_df

In [10]:
# Fill missing values in the 'Abstract' column with empty strings
news_df['Abstract'].fillna('', inplace=True)

# Fill missing values in the 'TitleEntities' column with empty strings
news_df['TitleEntities'].fillna('[]', inplace=True)

# Fill missing values in the 'AbstractEntities' column with empty strings
news_df['AbstractEntities'].fillna('[]', inplace=True)

In [11]:
news_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51282 entries, 0 to 51281
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   NewsID            51282 non-null  object
 1   Category          51282 non-null  object
 2   SubCategory       51282 non-null  object
 3   Title             51282 non-null  object
 4   Abstract          51282 non-null  object
 5   URL               51282 non-null  object
 6   TitleEntities     51282 non-null  object
 7   AbstractEntities  51282 non-null  object
dtypes: object(8)
memory usage: 3.1+ MB


## Knowledge Graph

In [12]:
graph_edges = []

for index, row in news_df.iterrows():
    news_id = row['NewsID']
    category = row['Category']
    subcategory = row['SubCategory']

    # Add edges for Category and SubCategory
    graph_edges.append({'source': news_id, 'relation': 'has_category', 'target': category})
    graph_edges.append({'source': news_id, 'relation': 'has_subcategory', 'target': subcategory})

In [13]:
import json

for index, row in news_df.iterrows():
    news_id = row['NewsID']
    title_entities_str = row['TitleEntities']
    abstract_entities_str = row['AbstractEntities']

    # Parse entity strings
    try:
        title_entities = json.loads(title_entities_str)
    except json.JSONDecodeError:
        title_entities = []

    try:
        abstract_entities = json.loads(abstract_entities_str)
    except json.JSONDecodeError:
        abstract_entities = []

    all_entities = title_entities + abstract_entities
    entity_labels = [entity['Label'] for entity in all_entities if 'Label' in entity]

    # Add edges for entities
    for entity_label in entity_labels:
        graph_edges.append({'source': news_id, 'relation': 'mentions_entity', 'target': entity_label})

In [14]:
from tqdm import tqdm

user_behaviors = {}

for index, row in tqdm(behaviors_df.iterrows(), total=behaviors_df.shape[0]):
    user_id = row['UserID']
    history = row['History']
    impressions = row['Impressions']

    # Extract clicked articles from History
    clicked_articles = history.split() if history else []

    # Analyze Impressions
    impression_list = impressions.split()
    clicked_impressions = [imp.split('-')[0] for imp in impression_list if imp.endswith('-1')]
    unclicked_impressions = [imp.split('-')[0] for imp in impression_list if imp.endswith('-0')]

    user_behaviors[user_id] = {'clicked_history': clicked_articles, 'clicked_impressions': clicked_impressions, 'unclicked_impressions': unclicked_impressions}

print(f'Processed behaviors for {len(user_behaviors)} users.')

100%|██████████| 156965/156965 [00:24<00:00, 6397.15it/s] 

Processed behaviors for 50000 users.


### Embeddings

In [15]:
# Load entity embeddings
entity_embeddings = {}
with open('/content/entity_embedding.vec', 'r') as f:
    next(f) # Skip header
    for line in f:
        values = line.strip().split('\t')
        entity = values[0]
        embedding = np.array(values[1:], dtype='float32')
        entity_embeddings[entity] = embedding

# Load relation embeddings
relation_embeddings = {}
with open('/content/relation_embedding.vec', 'r') as f:
    next(f) # Skip header
    for line in f:
        values = line.strip().split('\t')
        relation = values[0]
        embedding = np.array(values[1:], dtype='float32')
        relation_embeddings[relation] = embedding

In [16]:
# Create news embeddings
news_embeddings = {}
for index, row in tqdm(news_df.iterrows(), total=news_df.shape[0]):
    news_id = row['NewsID']
    title_entities_str = row['TitleEntities']
    abstract_entities_str = row['AbstractEntities']

    try:
        title_entities = json.loads(title_entities_str)
    except json.JSONDecodeError:
        title_entities = []

    try:
        abstract_entities = json.loads(abstract_entities_str)
    except json.JSONDecodeError:
        abstract_entities = []

    all_entities = title_entities + abstract_entities
    entity_labels = [entity['Label'] for entity in all_entities if 'Label' in entity]

    # Aggregate entity embeddings for news
    news_embedding = np.zeros(list(entity_embeddings.values())[0].shape)
    if entity_labels:
        entity_embeddings_list = [entity_embeddings[entity] for entity in entity_labels if entity in entity_embeddings]
        if entity_embeddings_list:
            news_embedding = np.mean(entity_embeddings_list, axis=0)

    news_embeddings[news_id] = news_embedding

print(f'Generated embeddings for {len(news_embeddings)} news articles.')

100%|██████████| 51282/51282 [00:32<00:00, 1554.77it/s]

Generated embeddings for 51282 news articles.


In [17]:
# Create user embeddings
user_embeddings = {}
for user_id, behaviors in tqdm(user_behaviors.items(), total=len(user_behaviors)):
    clicked_articles = behaviors['clicked_history'] + behaviors['clicked_impressions']

    # Aggregate news embeddings for user
    user_embedding = np.zeros(list(news_embeddings.values())[0].shape)
    if clicked_articles:
        clicked_news_embeddings = [news_embeddings[news_id] for news_id in clicked_articles if news_id in news_embeddings]
        if clicked_news_embeddings:
            user_embedding = np.mean(clicked_news_embeddings, axis=0)

    user_embeddings[user_id] = user_embedding

print(f'Generated embeddings for {len(user_embeddings)} users.')

100%|██████████| 50000/50000 [00:41<00:00, 1192.59it/s]

Generated embeddings for 50000 users.


## Recommendation Function

In [18]:
def recommend_news(user_id, news_embeddings, user_embeddings, user_behaviors, news_df, n=10):
    '''
    Recommends top N news articles for a given user.

    Args:
    user_id (str): The ID of the user.
    news_embeddings (dict): A dictionary mapping NewsID to its embedding.
    user_embeddings (dict): A dictionary mapping UserID to its embedding.
    user_behaviors (dict): A dictionary containing user behavior data.
    news_df (pd.DataFrame): DataFrame containing news information with 'NewsID' as index.
    n (int): The number of top recommendations to return.

    Returns:
        pd.DataFrame: A DataFrame containing 'NewsID', 'Title', and 'URL' of the recommended news.
    '''
    if user_id not in user_embeddings:
        return pd.DataFrame() # Return empty DataFrame if user embedding is not available

    user_embedding = user_embeddings[user_id]
    user_clicked_news = user_behaviors.get(user_id, {}).get('clicked_history', []) + user_behaviors.get(user_id, {}).get('clicked_impressions', [])

    # Calculate similarity scores
    similarity_scores = []
    for news_id, news_embedding in news_embeddings.items():
        if news_id not in user_clicked_news:
            similarity = cosine_similarity([user_embedding], [news_embedding])[0][0]
            similarity_scores.append((news_id, similarity))

    # Sort news articles by similarity score in descending order
    similarity_scores.sort(key=lambda x: x[1], reverse=True)

    # Get top N recommended news articles and their details
    recommended_news_details = []
    for news_id, score in similarity_scores[:n]:
        news_info = news_df[news_df['NewsID'] == news_id].iloc[0]
        recommended_news_details.append({'NewsID': news_id, 'Title': news_info['Title'], 'URL': news_info['URL']})

    return pd.DataFrame(recommended_news_details)

### Recommendations

In [19]:
# Example usage: Get recommendations for a user
example_user_id = list(user_embeddings.keys())[0] # Get the first user ID as an example
recommended_articles = recommend_news(example_user_id, news_embeddings, user_embeddings, user_behaviors, news_df)

print(f'Top 10 recommended articles for user {example_user_id}:')
print(recommended_articles)

Top 10 recommended articles for user U13740:
   NewsID                                              Title  \
0  N55528  The Brands Queen Elizabeth, Prince Charles, an...   
1  N19639                      50 Worst Habits For Belly Fat   
2  N61837  The Cost of Trump's Aid Freeze in the Trenches...   
3  N53526  I Was An NBA Wife. Here's How It Affected My M...   
4  N38324  How to Get Rid of Skin Tags, According to a De...   
5   N2073  Should NFL be able to fine players for critici...   
6  N49186  It's been Orlando's hottest October ever so fa...   
7  N59295  Chile: Three die in supermarket fire amid prot...   
8  N24510  Best PS5 games: top PlayStation 5 titles to lo...   
9  N39237     How to report weather-related closings, delays   

                                             URL  
0  https://assets.msn.com/labs/mind/AAGH0ET.html  
1  https://assets.msn.com/labs/mind/AAB19MK.html  
2  https://assets.msn.com/labs/mind/AAJgNsz.html  
3  https://assets.msn.com/labs/mind/AACk2N6.ht

## Save Models

In [20]:
import pickle

# Save user embeddings
with open('user_embeddings.pkl', 'wb') as f:
    pickle.dump(user_embeddings, f)
print('User embeddings saved to user_embeddings.pkl')

# Save news embeddings
with open('news_embeddings.pkl', 'wb') as f:
    pickle.dump(news_embeddings, f)
print('News embeddings saved to news_embeddings.pkl')

User embeddings saved to user_embeddings.pkl
News embeddings saved to news_embeddings.pkl
